# 03 — Customer Master Table & RFM Analysis

This is the hands-on RFM notebook. It keeps the exploratory approach used during practice and records the actual observed segment outputs.

**Goal:** create one row per persistent customer, calculate Recency/Frequency/Monetary, score customers and translate the scores into business segments.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data_loader import load_olist_data

data = load_olist_data(ROOT / "data_raw")
customers = data["customers"]
orders = data["orders"]
order_payments = data["payments"]

customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)



## Step 1 — Build customer-level paid value

Payment data can contain multiple payment rows for one order. Aggregate to order_id first so the join does not multiply order rows.


In [ ]:
order_value = (
    order_payments
    .groupby("order_id")["payment_value"]
    .sum()
    .reset_index()
    .rename(columns={"payment_value": "order_value"})
)

customer_orders_value = customer_orders.merge(
    order_value,
    on="order_id",
    how="left"
)



In [ ]:
customer_master = (
    customer_orders_value
    .groupby("customer_unique_id")
    .agg(
        first_purchase=("order_purchase_timestamp", "min"),
        last_purchase=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("order_value", "sum")
    )
    .reset_index()
)

customer_master["customer_lifetime_days"] = (
    customer_master["last_purchase"] - customer_master["first_purchase"]
).dt.days

analysis_date = customer_master["last_purchase"].max() + pd.Timedelta(days=1)
customer_master["recency"] = (
    analysis_date - customer_master["last_purchase"]
).dt.days

customer_master.shape



Observed customer master: **96,096 rows and 6 original columns**, one row per customer_unique_id. Recency is added as the next feature.


## Step 2 — Score Recency, Frequency and Monetary

- Recency: smaller is better.
- Frequency: larger is better.
- Monetary: larger is better.

Frequency is extremely skewed because most customers have one order. A business-rule score is used rather than forcing five quantile groups onto a highly concentrated variable.


In [ ]:
customer_master["R_score"] = pd.qcut(
    customer_master["recency"].rank(method="first"),
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

customer_master["M_score"] = pd.qcut(
    customer_master["monetary"].rank(method="first"),
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif x == 3:
        return 3
    elif x <= 5:
        return 4
    return 5

customer_master["F_score"] = customer_master["frequency"].apply(frequency_score)

customer_master["RFM_score"] = (
    customer_master["R_score"].astype(str)
    + customer_master["F_score"].astype(str)
    + customer_master["M_score"].astype(str)
)
customer_master["RFM_total"] = (
    customer_master["R_score"]
    + customer_master["F_score"]
    + customer_master["M_score"]
)

customer_master["RFM_total"].value_counts().sort_index()



## Step 3 — Create business segments

These are project-specific rule-based labels. They are not universal industry definitions.


In [ ]:
def rfm_segment(row):
    r, f, m = row["R_score"], row["F_score"], row["M_score"]

    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"
    elif r >= 3 and f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f <= 2:
        return "New / Recent Customers"
    elif r <= 2 and f >= 3:
        return "At Risk"
    elif r <= 2 and f <= 2:
        return "Lost / Inactive"
    elif m >= 4:
        return "High Value"
    return "Regular Customers"

customer_master["RFM_segment"] = customer_master.apply(rfm_segment, axis=1)



In [ ]:
segment_summary = (
    customer_master
    .groupby("RFM_segment")
    .agg(
        customers=("customer_unique_id", "nunique"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum")
    )
    .reset_index()
)

segment_summary["revenue_share_pct"] = (
    segment_summary["total_revenue"]
    / segment_summary["total_revenue"].sum()
    * 100
).round(2)

segment_summary.sort_values("total_revenue", ascending=False)



## Observed results from the practice run

| Segment | Customers | Share |
|---|---:|---:|
| New / Recent Customers | 38,481 | 40.04% |
| Lost / Inactive | 38,358 | 39.92% |
| Regular Customers | 11,453 | 11.92% |
| High Value | 7,552 | 7.86% |
| Loyal Customers | 144 | 0.15% |
| At Risk | 74 | 0.08% |
| Champions | 34 | 0.04% |

High Value customers were 7.86% of customers and contributed 13.40% of observed paid value in the earlier exploratory result.

**Interpretation:** the largest opportunity is converting recent first-time buyers into a second purchase. High monetary value alone does not imply loyalty.

**Reproducibility note:** this notebook records the earlier exploratory population. The new canonical pipeline uses delivered orders only, so its outputs may differ. Do not mix the two populations without stating the filter.


## Canonical RFM result from the supplied archive

The delivered-order customer master contains 93,358 customers. The largest groups are Lost / Inactive (37,272; 39.92%) and New / Recent Customers (37,222; 39.87%). High Value customers are 7,380 (7.91%) and contribute 13.55% of customer-paid value. The small Champions segment contains 32 customers. These are rule-based behavioral labels, not profit segments.

In [ ]:
rfm_summary = pd.read_csv(ROOT / "outputs" / "rfm_segment_summary.csv")
display(rfm_summary.sort_values("customers", ascending=False))